# Interfacing with VoteKit and abcvoting

`pref_voting` can convert its main election objects to and from the election objects of two other voting packages:

* **[VoteKit](https://votekit.readthedocs.io)** (MGGG): ranked ballots (with ties and truncation) and score ballots.
* **[abcvoting](https://abcvoting.readthedocs.io)** (Lackner et al.): approval-based committee (ABC) voting.

This lets you use, e.g., VoteKit's ballot generators with pref_voting's voting methods, pref_voting's profiles with VoteKit's elections, and pref_voting's approval profiles with abcvoting's committee rules.

Both packages are *optional* dependencies: a plain `pip install pref_voting` does not install them. To use the interfaces, install the extras:

```bash
uv pip install "pref_voting[votekit]"      # VoteKit (requires Python >= 3.11)
uv pip install "pref_voting[abcvoting]"    # abcvoting
uv pip install "pref_voting[interfaces]"   # both
```

The correspondence between the main election objects:

| pref_voting | VoteKit | abcvoting |
|---|---|---|
| `Profile` (linear orders) | `PreferenceProfile` of ranked ballots | — |
| `ProfileWithTies` (ties, truncation) | `PreferenceProfile` of ranked ballots | — |
| `GradeProfile` with numeric grades | `PreferenceProfile` of score ballots | — |
| `GradeProfile` with grades 0/1 (approval) | — | `Profile` |


In [1]:
from pref_voting.profiles import Profile
from pref_voting.profiles_with_ties import ProfileWithTies
from pref_voting.grade_profiles import GradeProfile
from pref_voting.utility_profiles import UtilityProfile

# the interface functions (also available as methods on the profile classes)
from pref_voting.interfaces.votekit_interface import (
    to_votekit_profile,
    votekit_to_profile_with_ties,
    grade_profile_to_votekit,
    votekit_to_grade_profile,
)
from pref_voting.interfaces.abcvoting_interface import (
    to_abcvoting_profile,
    abcvoting_to_grade_profile,
)

## From pref_voting to VoteKit

Any `Profile` or `ProfileWithTies` converts to a VoteKit `PreferenceProfile` of ranked ballots with `to_votekit()` (or the function `to_votekit_profile`). Candidates are named using the profile's `cmap`.

In [2]:
prof = Profile(
    [[0, 1, 2], [2, 1, 0], [1, 0, 2]],
    rcounts=[2, 3, 1],
    cmap={0: "Alice", 1: "Bob", 2: "Carol"},
)
prof.display()

vk_prof = prof.to_votekit()
vk_prof.df

+-------+-------+-------+
|   2   |   3   |   1   |
+-------+-------+-------+
| Alice | Carol |  Bob  |
|  Bob  |  Bob  | Alice |
| Carol | Alice | Carol |
+-------+-------+-------+


,Ranking_1,Ranking_2,Ranking_3,Voter Set,Weight
Ballot Index,,,,,
0,frozenset({Alice}),frozenset({Bob}),frozenset({Carol}),{},2.0
1,frozenset({Carol}),frozenset({Bob}),frozenset({Alice}),{},3.0
2,frozenset({Bob}),frozenset({Alice}),frozenset({Carol}),{},1.0


Now we can run any VoteKit election on the converted profile:

In [3]:
from votekit.elections import Borda, STV

election = Borda(vk_prof, n_seats=1)
print(election)

          Status  Round
Bob      Elected      1
Carol  Remaining      1
Alice  Remaining      1


## From VoteKit to pref_voting

The reverse direction is `ProfileWithTies.from_votekit` (or `votekit_to_profile_with_ties`). A typical use: generate profiles with VoteKit's ballot generators and analyze them with pref_voting's voting methods.

In [4]:
from votekit.ballot_generator import ic_profile_generator

vk_random = ic_profile_generator(candidates=["A", "B", "C", "D"], number_of_ballots=100)

prof_from_vk = ProfileWithTies.from_votekit(vk_random)
prof_from_vk.display()

+---+---+---+---+---+---+---+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+
| 8 | 5 | 4 | 4 | 9 | 4 | 4 | 11 | 4 | 5 | 4 | 3 | 4 | 5 | 2 | 4 | 2 | 5 | 1 | 3 | 4 | 2 | 2 | 1 |
+---+---+---+---+---+---+---+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+
| C | A | D | C | C | B | D | A  | D | C | B | A | B | C | A | D | C | D | D | B | A | A | B | B |
| B | D | C | A | B | A | B | C  | A | D | D | B | A | A | B | A | D | C | B | D | C | D | C | C |
| D | C | B | D | A | C | A | D  | B | B | C | D | D | B | C | C | A | A | C | A | B | B | A | D |
| A | B | A | B | D | D | C | B  | C | A | A | C | C | D | D | B | B | B | A | C | D | C | D | A |
+---+---+---+---+---+---+---+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+


In [5]:
from pref_voting.scoring_methods import borda
from pref_voting.margin_based_methods import split_cycle
from pref_voting.iterative_methods import instant_runoff

for vm in [borda, split_cycle, instant_runoff]:
    vm.display(prof_from_vk)

Borda winner is {C}
Split Cycle winners are {A, C}
Instant Runoff winners are {A, C}


Since these ballots are complete linear orders, the profile can also be converted to a `Profile` (candidates become the indices of the sorted candidate names, with the names retained in the `cmap`):

In [6]:
lin_prof = prof_from_vk.to_linear_profile()
lin_prof.display()

+---+---+---+---+---+---+---+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+
| 8 | 5 | 4 | 4 | 9 | 4 | 4 | 11 | 4 | 5 | 4 | 3 | 4 | 5 | 2 | 4 | 2 | 5 | 1 | 3 | 4 | 2 | 2 | 1 |
+---+---+---+---+---+---+---+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+
| C | A | D | C | C | B | D | A  | D | C | B | A | B | C | A | D | C | D | D | B | A | A | B | B |
| B | D | C | A | B | A | B | C  | A | D | D | B | A | A | B | A | D | C | B | D | C | D | C | C |
| D | C | B | D | A | C | A | D  | B | B | C | D | D | B | C | C | A | A | C | A | B | B | A | D |
| A | B | A | B | D | D | C | B  | C | A | A | C | C | D | D | B | B | B | A | C | D | C | D | A |
+---+---+---+---+---+---+---+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+


## Ties and truncation

VoteKit rankings are lists of *sets* of candidates: candidates in the same set are tied, and candidates missing from a ballot are unranked. This matches `ProfileWithTies` exactly, so profiles with ties and truncated ballots round-trip through VoteKit.

In [7]:
prof_ties = ProfileWithTies(
    [
        {"a": 1, "b": 1, "c": 2},   # a and b tied for first, c second
        {"c": 1},                   # only c is ranked
        {"b": 1, "a": 2},           # truncated: c unranked
    ],
    rcounts=[2, 1, 3],
    candidates=["a", "b", "c"],
)
prof_ties.display()

vk_ties = prof_ties.to_votekit()
vk_ties.df

+-----+---+---+
|  2  | 1 | 3 |
+-----+---+---+
| a b | c | b |
|  c  |   | a |
+-----+---+---+


,Ranking_1,Ranking_2,Ranking_3,Voter Set,Weight
Ballot Index,,,,,
0,"frozenset({b, a})",frozenset({c}),frozenset({~}),{},2.0
1,frozenset({c}),frozenset({~}),frozenset({~}),{},1.0
2,frozenset({b}),frozenset({a}),frozenset({~}),{},3.0


In [8]:
back = ProfileWithTies.from_votekit(vk_ties)
back.display()

+-----+---+---+
|  2  | 1 | 3 |
+-----+---+---+
| b a | c | b |
|  c  |   | a |
+-----+---+---+


## Ballot weights

VoteKit ballots carry weights, which may be fractional; pref_voting profiles count *voters*, so `rcounts` are integers. Integer-valued weights convert directly. Fractional weights raise an error unless you pass `scale_weights=True`, which multiplies all weights by a common denominator — an exact rescaling that preserves all margins and scores, but changes the total number of voters.

In [9]:
from votekit.ballot import Ballot
from votekit.pref_profile import PreferenceProfile

vk_frac = PreferenceProfile(
    ballots=(
        Ballot(ranking=[{"a"}, {"b"}], weight=0.5),
        Ballot(ranking=[{"b"}, {"a"}], weight=1.5),
    ),
    candidates=("a", "b"),
)

try:
    ProfileWithTies.from_votekit(vk_frac)
except ValueError as err:
    print(f"ValueError: {err}")

ValueError: The profile contains non-integer ballot weights, but pref_voting profiles use integer counts. Pass scale_weights=True to multiply all weights by a common denominator (this changes the total number of voters but not the relative weights of the ballots).


In [10]:
scaled = ProfileWithTies.from_votekit(vk_frac, scale_weights=True)
scaled.display()  # weights 0.5 and 1.5 become counts 1 and 3

+---+---+
| 1 | 3 |
+---+---+
| a | b |
| b | a |
+---+---+


## Grade profiles and VoteKit score ballots

A `GradeProfile` with numeric grades corresponds to a VoteKit `PreferenceProfile` of *score* ballots.

**Caveat:** VoteKit does not store scores of 0, so a candidate assigned the grade 0 becomes an *unscored* candidate in VoteKit. Keep this in mind when a 0 grade is meaningful (e.g., explicit disapproval).

In [11]:
gprof = GradeProfile(
    [
        {"x": 3, "y": 1, "z": 0},
        {"x": 2, "y": 2},          # z is not graded
        {"y": 3, "z": 1},
    ],
    [0, 1, 2, 3],
    gcounts=[2, 1, 1],
    candidates=["x", "y", "z"],
)
gprof.display()

vk_scores = gprof.to_votekit()
vk_scores.df

      2  1    1
--  ---  ---  ---
x     3  2
y     1  2    3
z     0       1


,x,y,z,Voter Set,Weight
Ballot Index,,,,,
0,3.0,1.0,NaN,{},2.0
1,2.0,2.0,NaN,{},1.0
2,NaN,3.0,1.0,{},1.0


In [12]:
# coming back, pass the intended grade scale (a VoteKit profile only carries the
# assigned values, not the scale)
back_gprof = GradeProfile.from_votekit(vk_scores, grades=[0, 1, 2, 3])
back_gprof.display()  # note: the 0 assigned to z on the first ballot was not preserved

    2    1    1
--  ---  ---  ---
x   3    2
y   1    2    3
z             1


## From pref_voting to abcvoting

abcvoting implements approval-based committee (ABC) rules. Its `Profile` is a profile of *approval ballots*, which corresponds to a `GradeProfile` with grades 0 and 1: a voter approves a candidate by assigning grade 1.

abcvoting identifies candidates with integers $0, \ldots, m - 1$; the conversion preserves candidate names via `cand_names`. Ballot counts become abcvoting voter weights.

In [13]:
approval_prof = GradeProfile(
    [
        {"w": 1, "x": 1, "y": 0, "z": 0},
        {"w": 0, "x": 1, "y": 1, "z": 0},
        {"w": 0, "x": 0, "y": 1, "z": 1},
        {"w": 1, "x": 0, "y": 0, "z": 1},
    ],
    [0, 1],
    gcounts=[3, 4, 2, 1],
    candidates=["w", "x", "y", "z"],
)
approval_prof.display()

abc_prof = approval_prof.to_abcvoting()
print(abc_prof)

      3    4    2    1
--  ---  ---  ---  ---
w     1    0    0    1
x     1    1    0    0
y     0    1    1    0
z     0    0    1    1
weighted profile with 4 voters and 4 candidates:
 voter 0:   3 * {w, x},
 voter 1:   4 * {x, y},
 voter 2:   2 * {y, z},
 voter 3:   1 * {w, z}


Now any abcvoting rule can be applied. `abcrules.compute` takes a rule identifier and a committee size:

In [14]:
from abcvoting import abcrules

for rule_id in ["av", "pav", "seqphragmen", "equal-shares"]:
    committees = abcrules.compute(rule_id, abc_prof, committeesize=2, resolute=True)
    named = [[abc_prof.cand_names[c] for c in sorted(committee)] for committee in committees]
    print(f"{abcrules.Rule(rule_id).shortname:>15}: {named}")

             AV: [['x', 'y']]
            PAV: [['x', 'y']]
   seq-Phragmén: [['x', 'y']]
   Equal Shares: [['x', 'y']]


Ballot counts become abcvoting voter *weights*. Most ABC rules support weighted profiles, but a few (e.g., Monroe) are only defined for unit weights. abcvoting can expand a weighted profile into one voter per ballot:

In [15]:
unit_prof = approval_prof.to_abcvoting()
unit_prof.convert_to_unit_weights()
print(f"number of voters after expanding weights: {len(unit_prof)}")

committees = abcrules.compute("monroe", unit_prof, committeesize=2, resolute=True)
print("Monroe:", [[unit_prof.cand_names[c] for c in sorted(committee)] for committee in committees])

number of voters after expanding weights: 10
Monroe: [['x', 'y']]


## From abcvoting to pref_voting

An abcvoting profile converts losslessly to a `GradeProfile` with grades 0 and 1 (approved candidates get grade 1, all others 0, voter weights become ballot counts):

In [16]:
from abcvoting.preferences import Profile as AbcvotingProfile, Voter

abc = AbcvotingProfile(3, cand_names=["a", "b", "c"])
abc.add_voters([[0, 1], [1, 2], [0], [0, 2]])

gprof_from_abc = GradeProfile.from_abcvoting(abc)
gprof_from_abc.display()
print("Approval scores:", gprof_from_abc.approval_scores())

      1    1    1    1
--  ---  ---  ---  ---
a     1    0    1    1
b     1    1    0    0
c     0    1    0    1
Approval scores: {0: np.int64(3), 1: np.int64(2), 2: np.int64(2)}


## End to end: utilities → approval ballots → abcvoting

pref_voting's cardinal objects already provide the natural bridges to approval ballots: `UtilityProfile.to_approval_profile()` and `to_k_approval_profile(k)` return `GradeProfile`s with grades 0 and 1. Combined with the interface, this gives a pipeline from pref_voting's generators to abcvoting's committee rules:

In [17]:
from pref_voting.generate_utility_profiles import generate_utility_profile_uniform

uprof = generate_utility_profile_uniform(5, 20)      # 5 candidates, 20 voters
k_approval = uprof.to_k_approval_profile(2)          # each voter approves their top 2

committees = abcrules.compute("pav", k_approval.to_abcvoting(), committeesize=3, resolute=True)
print("PAV committee:", sorted(committees[0]))

PAV committee: [0, 1, 3]


## Summary

| conversion | function | method |
|---|---|---|
| `Profile` / `ProfileWithTies` → VoteKit | `to_votekit_profile` | `.to_votekit()` |
| VoteKit ranked → `ProfileWithTies` | `votekit_to_profile_with_ties` | `ProfileWithTies.from_votekit` |
| `GradeProfile` → VoteKit scores | `grade_profile_to_votekit` | `.to_votekit()` |
| VoteKit scores → `GradeProfile` | `votekit_to_grade_profile` | `GradeProfile.from_votekit` |
| `GradeProfile` (0/1) → abcvoting | `to_abcvoting_profile` | `.to_abcvoting()` |
| abcvoting → `GradeProfile` | `abcvoting_to_grade_profile` | `GradeProfile.from_abcvoting` |

Conversion policies to remember:

* **Weights**: integer-valued weights convert directly; fractional weights raise unless `scale_weights=True` (exact rescaling).
* **abcvoting** requires a `GradeProfile` with grades 0 and 1; use `UtilityProfile.to_approval_profile()` / `to_k_approval_profile(k)` to produce one.
* **VoteKit score ballots** require numeric grades, and VoteKit does not store scores of 0.
